# Kafka Foundations & Architecture

## What's covered

- What Kafka is and where it earns its complexity over a database or a message queue
- Cluster anatomy: brokers, producers, consumers, the controller
- The append-only log — the one data structure the rest of Kafka is built on
- Topics, partitions, offsets — the three nouns the whole API is shaped around
- Replication, leaders, followers, and the in-sync replicas (ISR) set
- The controller — ZooKeeper vs KRaft, and why ZooKeeper is on the way out
- Hosting options — self-managed, Amazon MSK, Confluent Cloud, Redpanda
- When NOT to use Kafka
- Setup — the canonical `confluent-kafka` client pattern reused throughout this series
- Hello Kafka — create a topic, produce a few records, consume them back
- Ordering guarantees — what Kafka promises and what it does not
- Delivery semantics primer — at-most-once, at-least-once, exactly-once (full treatment in notebooks 02 and 03)

## The problem Kafka solves

Imagine a bank where the mobile app, the fraud detector, the ledger, the analytics warehouse, and the customer-notification service all need to know about every payment the moment it happens. With five services, you could wire each one directly to the source — that's ten point-to-point connections. With twenty services, it's one hundred ninety. Each connection has its own retry logic, its own back-pressure handling, its own schema assumptions. One slow consumer back-pressures the producer. One schema change breaks five integrations.

Kafka exists because this pattern — many producers, many consumers, many event types, all needing a durable buffer in between — keeps appearing the moment a company outgrows a single database. Instead of point-to-point wiring, every event is appended to a shared log, and every service reads from that log at its own pace.

## What Kafka actually is

Apache Kafka is a **distributed, append-only log**. Producers append records to the end. Consumers read records in order, tracking their own position. Records are not deleted when consumed — they stay until a retention policy (time or size) removes them. That single design choice is why one Kafka cluster can simultaneously feed a real-time fraud detector, a nightly batch warehouse load, and a brand-new microservice that wasn't even written when the events landed.

The mental shift from a traditional message queue: a queue is destructive — once a consumer reads a message, it's gone. Kafka is non-destructive — the log is the source of truth, and consumers are just bookmarks pointing into it. From a database: a database is built for random reads and updates; Kafka is built for sequential appends and sequential reads. Different shape, different physics.

## The cluster — brokers, producers, consumers, the controller

Four roles whenever Kafka runs:

- **Broker** — a server that stores log segments on disk and serves produce/fetch requests. A Kafka cluster is a set of brokers (typically three or more for production). Brokers are interchangeable from a client's perspective; any broker can route a client to the right partition leader.
- **Producer** — a client that appends records to a topic. Your application code.
- **Consumer** — a client that reads records from a topic. Also your application code, on the other side.
- **Controller** — one elected broker that coordinates cluster-wide metadata: who leads which partition, which replicas are in-sync, what topics exist. In modern Kafka the controller is part of the broker set itself (KRaft mode); in legacy clusters it was managed via ZooKeeper.

```text
                          ┌────────────────────────────────────┐
                          │           Kafka cluster            │
                          │   ┌────────┐ ┌────────┐ ┌────────┐ │
  ┌──────────┐  produce   │   │Broker 1│ │Broker 2│ │Broker 3│ │
  │ Producer │ ─────────► │   │  ★ ctrl│ │        │ │        │ │
  └──────────┘            │   └────────┘ └────────┘ └────────┘ │
                          │     replicated topic partitions    │
  ┌──────────┐  fetch     │                                    │
  │ Consumer │ ◄───────── │                                    │
  └──────────┘            └────────────────────────────────────┘
```

Producers and consumers don't talk to each other. They both talk to the cluster, and the log is what's between them.

## The append-only log — one data structure, everything else falls out of it

Every Kafka partition is, on disk, an append-only file. New records go to the end. Each record gets a monotonically increasing **offset** — `0, 1, 2, 3, ...` — assigned when it lands. Reads start at an offset and stream forward sequentially.

```text
  partition (on disk)
  ┌──────┬──────┬──────┬──────┬──────┬──────┬──────┐
  │  0   │  1   │  2   │  3   │  4   │  5   │  6   │ ◄── append here
  └──────┴──────┴──────┴──────┴──────┴──────┴──────┘
    ▲                  ▲           ▲
    │                  │           │
  retention      consumer A     consumer B
   edge          offset = 3     offset = 5
```

Three things fall out of this choice:

- **Sequential I/O is fast.** A single modern disk can sustain hundreds of megabytes per second of sequential writes. Random I/O would be a hundred times slower. Kafka is fast because it picked the workload pattern the hardware was already good at.
- **Reads don't interfere with writes.** Consumers are at different positions and never block each other or the producer. The kernel's page cache serves recent reads from RAM without Kafka having to manage a cache itself.
- **Replay is trivial.** Want a new consumer to reprocess last week's data? Start from offset zero. The log doesn't care.

## Topics, partitions, offsets

These three nouns shape the whole API:

- **Topic** — a named stream of records, like `payments` or `clicks`. Producers write to it, consumers read from it.
- **Partition** — a topic is split into one or more partitions. Each partition is an independent append-only log on a specific broker. Partitions are the unit of parallelism: more partitions means more consumers can read in parallel.
- **Offset** — the position of a record within a partition. Unique only inside its partition — offset 5 in partition 0 and offset 5 in partition 1 are different records.

```text
  topic: payments    (3 partitions)

  partition 0  ┌──┬──┬──┬──┬──┬──┐
               │0 │1 │2 │3 │4 │5 │
               └──┴──┴──┴──┴──┴──┘
  partition 1  ┌──┬──┬──┬──┐
               │0 │1 │2 │3 │
               └──┴──┴──┴──┘
  partition 2  ┌──┬──┬──┬──┬──┬──┬──┬──┐
               │0 │1 │2 │3 │4 │5 │6 │7 │
               └──┴──┴──┴──┴──┴──┴──┴──┘
```

**The ordering rule you must remember:** Kafka guarantees ordering **within a partition**, not across partitions. If two records must be processed in the order they were produced, they must land on the same partition. The producer controls that by choosing a record **key** — records with the same key always hash to the same partition. No key? The producer picks a partition round-robin and ordering across the topic is not guaranteed.

For a `payments` topic, a natural key is `customer_id` — every payment for the same customer ends up on the same partition, in order, and one consumer can process that customer's history sequentially.

## Replication, leaders, followers, ISR

A single disk fails. To survive that, each partition is **replicated** across multiple brokers. The **replication factor** is the number of copies — three is the common production choice.

Among the replicas, one is the **leader**. All produce and fetch traffic for that partition goes through the leader. The other replicas are **followers**; they continuously pull from the leader and stay in step.

```text
  topic: payments, partition 0, replication factor = 3

       producer ──► leader (Broker 1) ──► writes to local log
                          │
             replicates   ├──► follower (Broker 2)  ──► in-sync
                          └──► follower (Broker 3)  ──► in-sync
                                  ▲
                                  │
                                  └── if a follower falls behind by more than
                                      replica.lag.time.max.ms, it drops out
                                      of the ISR set
```

The **in-sync replicas (ISR)** set is the subset of replicas currently caught up with the leader. If a follower falls more than `replica.lag.time.max.ms` behind, it's removed from the ISR. When the leader dies, a new leader is elected from the ISR — so a record is only safe once it's been written to all replicas in the ISR.

The producer config `acks=all` (covered in notebook 02) means "don't acknowledge a write until every ISR replica has it." Combined with a broker-level `min.insync.replicas=2`, you get the standard durable-write recipe: a write that returns successfully has survived at least two replicas, so a single broker death cannot lose it.

## The controller — ZooKeeper vs KRaft

Somebody has to decide which broker leads partition 0 of `payments`, what to do when broker 2 disappears, and how the cluster agrees on its own membership. That somebody is the **controller**.

Two generations of how Kafka handles this:

- **ZooKeeper mode (legacy).** Cluster metadata lived in an external ZooKeeper ensemble — a separate distributed system you also had to operate. One broker was elected controller and synced state via ZooKeeper. This worked, but it meant two clusters to monitor, two failure domains, and a metadata bottleneck that limited how many partitions one cluster could host.
- **KRaft mode (modern).** Kafka uses its own Raft-based consensus protocol — KRaft, for *Kafka Raft* — to manage metadata internally. A small group of brokers (typically three or five) play the **controller quorum** role alongside their normal broker duties. No external system, faster failover, and clusters can scale to millions of partitions.

KRaft became production-ready in **Kafka 3.3** (2022). Kafka 4.0 removes ZooKeeper entirely. If you're spinning up a new cluster today, it's KRaft — and the exam expects you to know the difference and the direction of travel.

## Hosting options

How a cluster actually shows up in your environment:

- **Self-managed (Apache Kafka).** Run brokers on your own VMs, containers, or Kubernetes. Maximum control, maximum operational cost.
- **Amazon MSK (Managed Streaming for Kafka).** AWS runs the brokers and ZooKeeper/KRaft quorum; you point clients at a bootstrap endpoint. There's also **MSK Serverless** which hides broker count entirely and charges per-throughput.
- **Confluent Cloud.** The commercial managed offering from Confluent, the company founded by Kafka's original creators. Includes Schema Registry, Connect, and ksqlDB as managed services on top.
- **Redpanda.** A drop-in Kafka-API-compatible engine written in C++ instead of Java. No JVM, no separate ZooKeeper. Different implementation, same protocol — the clients in this notebook work unchanged.

For learning, a single broker in Docker (the local setup) is plenty. For the exam, you should at minimum recognize each name.

## When NOT to use Kafka

Before standing up a cluster, a word on when Kafka is the wrong tool:

- **You have one producer and one consumer.** A direct HTTP call or a database table is simpler. Kafka earns its complexity when fan-out, decoupling, or durability across services starts to hurt.
- **You need request/response semantics.** Kafka is fire-and-forget on the producer side and pull-based on the consumer side. Use gRPC or REST for synchronous calls.
- **Your records are huge.** Kafka is tuned for records in the kilobyte range. Multi-megabyte payloads are technically allowed but bad citizens — store the payload in object storage (S3, GCS) and put a pointer record on the topic.
- **You need a transactional database.** Kafka has transactions, but they're scoped to atomic produce-plus-offset-commit across topics. They are not a substitute for ACID over arbitrary mutable state. Reach for Postgres.
- **You need sub-millisecond, single-record lookups by key.** Kafka is a log, not a key-value store. Pair it with Redis, RocksDB, or a database for that access pattern.

The honest rule: Kafka earns its complexity when you have multiple independent consumers of the same event stream, or when durable buffering between services starts solving more problems than it creates.

## Setup — your first Kafka client

Run a single-broker cluster in Docker (the simplest path):

```bash
docker run -d --name kafka -p 9092:9092 apache/kafka:3.8.0
```

Install the Python client:

```bash
pip install confluent-kafka==2.5.3
```

`confluent-kafka` is a thin Python wrapper around `librdkafka`, the C client maintained by Confluent. It is the recommended Python client — faster and more feature-complete than the older pure-Python `kafka-python` package.

The pattern below is what every notebook in the series uses: configure a bootstrap server, build a client, do work. The **bootstrap server** is the initial address the client connects to; from there it discovers the rest of the cluster on its own — you only need to list one or two brokers, not all of them.

In [ ]:
import confluent_kafka

BOOTSTRAP = "localhost:9092"

print("confluent-kafka  :", confluent_kafka.version()[0])
print("librdkafka       :", confluent_kafka.libversion()[0])
print("bootstrap server :", BOOTSTRAP)

## Hello Kafka — create a topic, produce, consume

The minimum loop end-to-end. Three steps:

1. Use `AdminClient` to create a small demo topic with three partitions.
2. Use `Producer` to send a handful of keyed records.
3. Use `Consumer` to read them back, printing the partition and offset each record landed on.

Watch the partition column in the consumer output — records with the same `customer_id` key consistently land on the same partition, while different keys spread across partitions zero through two.

In [ ]:
from confluent_kafka.admin import AdminClient, NewTopic

TOPIC = "foundations-demo"

admin = AdminClient({"bootstrap.servers": BOOTSTRAP})

# Create the topic with 3 partitions and replication factor 1 (single-broker dev cluster).
# create_topics returns a dict of futures — one per topic — so we wait on the result.
futures = admin.create_topics([NewTopic(TOPIC, num_partitions=3, replication_factor=1)])
for name, fut in futures.items():
    try:
        fut.result()
        print(f"Created topic: {name}")
    except Exception as e:
        # 'TopicExistsError' is fine on a re-run.
        print(f"{name}: {e}")

In [ ]:
from confluent_kafka import Producer

producer = Producer({"bootstrap.servers": BOOTSTRAP})

payments = [
    ("CUST0001", "deposit  120.00"),
    ("CUST0002", "withdraw  40.00"),
    ("CUST0001", "transfer  75.00"),   # same key as the first record
    ("CUST0003", "deposit  500.00"),
    ("CUST0002", "deposit   25.00"),   # same key as the second record
    ("CUST0001", "withdraw  10.00"),   # same key as the first record
]

# produce() is asynchronous — it just enqueues. The client batches in the background.
for key, value in payments:
    producer.produce(TOPIC, key=key, value=value)

# flush() blocks until every queued message is acknowledged by the broker.
producer.flush()
print(f"Produced {len(payments)} records to {TOPIC}")

In [ ]:
from confluent_kafka import Consumer

consumer = Consumer({
    "bootstrap.servers": BOOTSTRAP,
    "group.id": "foundations-demo-reader",
    "auto.offset.reset": "earliest",   # start at offset 0 if this group is new
    "enable.auto.commit": False,       # we won't commit offsets in this demo
})
consumer.subscribe([TOPIC])

print(f"{'partition':>9}  {'offset':>6}  {'key':<10}  value")
print("-" * 50)

# Poll a few times. Each poll returns at most one record; None means no record was ready.
received = 0
while received < len(payments):
    msg = consumer.poll(timeout=2.0)
    if msg is None:
        break  # no more records arriving — exit rather than hang
    if msg.error():
        print("ERROR:", msg.error())
        continue
    print(f"{msg.partition():>9}  {msg.offset():>6}  {msg.key().decode():<10}  {msg.value().decode()}")
    received += 1

consumer.close()

## Ordering guarantees, in one place

It's worth stating the rules in one block so you can carry them as a checklist:

- **Within a partition:** records are strictly ordered by offset, both on disk and as delivered to consumers. Always.
- **Across partitions of the same topic:** no ordering guarantee. Two records produced in quick succession to different partitions may be consumed in either order.
- **With a key:** records with the same key always go to the same partition (assuming the default partitioner and that the partition count hasn't changed since the records were produced). So same-key records are strictly ordered.
- **Without a key:** the producer round-robins across partitions. Ordering across the topic is not guaranteed.
- **After a partition count change:** the key-to-partition mapping changes. Records produced before the change and records produced after may live on different partitions even for the same key. Adding partitions to a keyed topic breaks per-key ordering for keys that move.

Design implication: choose the key for what you need to order by. For `payments`, key on `customer_id` so each customer's history is ordered. For `clicks`, key on `session_id` if you need clickstream order per session.

## Delivery semantics — the three options

Every messaging system has to answer: if something fails between producing and consuming, what do we promise about each record? Kafka gives you three settings, tunable end to end:

- **At most once.** Records may be lost, never duplicated. Producer fires and forgets; consumer commits its offset before processing. Rare in practice — almost nobody wants silent data loss.
- **At least once.** Records are never lost, but may be delivered more than once after a retry or a consumer crash. The default behavior. Downstream code must be **idempotent** — re-processing the same record twice must yield the same result. This is the workhorse mode.
- **Exactly once.** Each record is processed exactly once, even with broker, producer, or consumer failures. Requires the **idempotent producer** plus **transactions** plus a transaction-aware consumer (`isolation.level=read_committed`). Strict, but more expensive and more constrained.

Notebook 02 covers the producer side — idempotence, `acks`, transactions. Notebook 03 covers the consumer side — offset commits, rebalances, `isolation.level`. For now, the takeaway: **at-least-once with idempotent downstream code is the default, exactly-once is opt-in for the cases that need it.**

## What's next

This notebook set the vocabulary: brokers, partitions, offsets, replicas, the ISR, KRaft, and the at-least-once default. From here the curriculum drills into each surface:

- **02 Producers** — keys, partitioners, `acks`, idempotence, transactions, batching and `linger.ms`, compression.
- **03 Consumers & Consumer Groups** — groups and rebalancing, offset commits, `auto.offset.reset`, assignment strategies, the read-committed isolation level.
- **04 Topics, Partitions & Storage** — sizing partition counts, retention policy, log compaction, segment files on disk.

Every later notebook assumes you can answer: *what is a partition, what is an offset, what is the ISR, and what does Kafka order?* If those land, the rest is mechanism.